# 为什么逐 chunk 的 `<think>` 标签过滤会失败？

本 notebook 演示在流式输出场景下，**逐 chunk 对 `<think>...</think>` 做正则替换**为什么无效，
以及 `openai_client.py` 中基于缓冲区（buffer）的方案是如何修复这个问题的。

## 背景

部分 OpenAI-compatible 模型（MiniMax-M2.5、DeepSeek-R1 等）会把推理过程直接嵌在 `delta.content` 里：

```
<think>
这里是模型的内部推理，不应该展示给用户...
</think>
这里才是真正要展示给用户的回答。
```

问题在于：这段内容是**流式**到达的，被分成许多小 chunk，而标签可能横跨多个 chunk。

## 1. 模拟流式 chunk

我们用一段真实感的响应模拟 SSE streaming，故意把 `<think>` 标签切割在不同 chunk 边界处。

In [1]:
# 完整的模型响应（用户不应看到 <think>…</think> 部分）
FULL_RESPONSE = (
    "<think>\n"
    "用户问的是 Python 中的列表推导式。\n"
    "我应该给一个简洁的例子。\n"
    "</think>\n"
    "列表推导式是一种简洁的创建列表的语法：\n"
    "```python\n"
    "squares = [x**2 for x in range(10)]\n"
    "```"
)

print("完整响应:")
print(repr(FULL_RESPONSE))
print()
print("用户应该看到的内容:")
import re
print(re.sub(r'<think>.*?</think>', '', FULL_RESPONSE, flags=re.DOTALL))

完整响应:
'<think>\n用户问的是 Python 中的列表推导式。\n我应该给一个简洁的例子。\n</think>\n列表推导式是一种简洁的创建列表的语法：\n```python\nsquares = [x**2 for x in range(10)]\n```'

用户应该看到的内容:

列表推导式是一种简洁的创建列表的语法：
```python
squares = [x**2 for x in range(10)]
```


In [2]:
def simulate_chunks(text: str, sizes: list[int]) -> list[str]:
    """按指定大小切割字符串，模拟 SSE 流式 chunk。"""
    chunks = []
    pos = 0
    for size in sizes:
        chunk = text[pos:pos+size]
        if chunk:
            chunks.append(chunk)
        pos += size
    if pos < len(text):
        chunks.append(text[pos:])
    return chunks

# 模拟真实 LLM 流式输出：
#  - <think>\n 作为完整 token 一次发送（LLM tokenizer 通常整体 emit）
#  - 推理内容按自然语言边界切割
#  - </think> 被切割在 chunk 边界（这是实际场景中最常见的问题）
# FULL_RESPONSE 字符串各位置:
#   0  '<think>\n'          (8字符)
#   8  '用户问的是 Python'    (14字符, 包含'\n')
#   22 ' 中的列表推'          (若干)
#   ...  最终 '</thi' 和 'nk>' 被分到不同 chunk
chunk_sizes = [8, 14, 9, 14, 13, 6, 4, 100]  # </think> 被 6+4 切割

CHUNKS = simulate_chunks(FULL_RESPONSE, chunk_sizes)

print(f"共 {len(CHUNKS)} 个 chunk:")
for i, c in enumerate(CHUNKS):
    print(f"  chunk[{i:2d}]: {repr(c)}")

# 验证切割后确实有 </think> 跨 chunk
full_joined = ''.join(CHUNKS)
assert full_joined == FULL_RESPONSE, '切割后拼接应还原原文'
print('\n<think> 是否在单个 chunk 中完整:', any('<think>' in c for c in CHUNKS))
print('</think> 是否被切割跨 chunk:', not any('</think>' in c for c in CHUNKS))

共 8 个 chunk:
  chunk[ 0]: '<think>\n'
  chunk[ 1]: '用户问的是 Python 中'
  chunk[ 2]: '的列表推导式。\n我'
  chunk[ 3]: '应该给一个简洁的例子。\n</'
  chunk[ 4]: 'think>\n列表推导式是'
  chunk[ 5]: '一种简洁的创'
  chunk[ 6]: '建列表的'
  chunk[ 7]: '语法：\n```python\nsquares = [x**2 for x in range(10)]\n```'

<think> 是否在单个 chunk 中完整: True
</think> 是否被切割跨 chunk: True


## 2. 旧方案（逐 chunk 正则替换）——失败

`openai_client.py` 在添加缓冲区之前，每收到一个 chunk 就直接 yield 给 TUI：

```python
# 旧代码（d5c8888 commit 状态，无 think 过滤）
if delta.content:
    collected_content += delta.content
    yield ApiTextDeltaEvent(text=delta.content)   # 直接透传，<think> 内容泄漏给用户
```

有人可能会想到「在 yield 前加一行正则替换」来修复：

```python
# 看似合理的「修复」
if delta.content:
    visible = re.sub(r'<think>.*?</think>', '', delta.content, flags=re.DOTALL)
    if visible:
        yield ApiTextDeltaEvent(text=visible)
```

下面演示这为什么**不能**工作：

In [3]:
import re

THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL)

def naive_strip_per_chunk(chunks: list[str]) -> str:
    """旧方案：每个 chunk 单独做正则替换，然后直接输出。"""
    output_parts = []
    for i, chunk in enumerate(chunks):
        visible = THINK_RE.sub('', chunk)
        output_parts.append(visible)
        if visible != chunk:
            print(f"  chunk[{i:2d}]: 替换生效  {repr(chunk)} → {repr(visible)}")
        else:
            print(f"  chunk[{i:2d}]: 未命中    {repr(chunk)}")
    return ''.join(output_parts)

print("=== 逐 chunk 正则替换过程 ===")
naive_result = naive_strip_per_chunk(CHUNKS)
print()
print("=== 用户实际看到的内容 ===")
print(repr(naive_result))
print()
print("包含 <think>？", '<think>' in naive_result)

=== 逐 chunk 正则替换过程 ===
  chunk[ 0]: 未命中    '<think>\n'
  chunk[ 1]: 未命中    '用户问的是 Python 中'
  chunk[ 2]: 未命中    '的列表推导式。\n我'
  chunk[ 3]: 未命中    '应该给一个简洁的例子。\n</'
  chunk[ 4]: 未命中    'think>\n列表推导式是'
  chunk[ 5]: 未命中    '一种简洁的创'
  chunk[ 6]: 未命中    '建列表的'
  chunk[ 7]: 未命中    '语法：\n```python\nsquares = [x**2 for x in range(10)]\n```'

=== 用户实际看到的内容 ===
'<think>\n用户问的是 Python 中的列表推导式。\n我应该给一个简洁的例子。\n</think>\n列表推导式是一种简洁的创建列表的语法：\n```python\nsquares = [x**2 for x in range(10)]\n```'

包含 <think>？ True


**关键观察**：每个 chunk 都没有完整的 `<think>...</think>` 配对，所以正则从未命中，
`<think>` 内容完整地泄漏给了用户。

这和 `re.DOTALL` 没有关系——问题在于**正则需要同时看到开标签和闭标签**，
而流式切割让它们落在不同的 chunk 里。

## 3. 边界情况分析

失败的根本原因有两种：

### 情况 A：闭标签跨 chunk

In [4]:
case_a_chunks = [
    "<think>some reasoning</",   # 闭标签被截断
    "think> visible text",       # 闭标签的后半部分在下一 chunk
]

print("=== 情况 A：闭标签跨 chunk ===")
for i, c in enumerate(case_a_chunks):
    result = THINK_RE.sub('', c)
    print(f"  chunk[{i}] 输入: {repr(c)}")
    print(f"  chunk[{i}] 输出: {repr(result)}")
    print()

=== 情况 A：闭标签跨 chunk ===
  chunk[0] 输入: '<think>some reasoning</'
  chunk[0] 输出: '<think>some reasoning</'

  chunk[1] 输入: 'think> visible text'
  chunk[1] 输出: 'think> visible text'



In [5]:
# 情况 B：开标签跨 chunk
case_b_chunks = [
    "some text <thi",            # 开标签被截断
    "nk>reasoning</think> ok",   # 开标签后半部分在下一 chunk
]

print("=== 情况 B：开标签跨 chunk ===")
naive_b = []
for i, c in enumerate(case_b_chunks):
    result = THINK_RE.sub('', c)
    naive_b.append(result)
    print(f"  chunk[{i}] 输入: {repr(c)}")
    print(f"  chunk[{i}] 输出: {repr(result)}")
    print()

print("合并输出:", repr(''.join(naive_b)))

=== 情况 B：开标签跨 chunk ===
  chunk[0] 输入: 'some text <thi'
  chunk[0] 输出: 'some text <thi'

  chunk[1] 输入: 'nk>reasoning</think> ok'
  chunk[1] 输出: 'nk>reasoning</think> ok'

合并输出: 'some text <think>reasoning</think> ok'


## 4. 当前方案（缓冲区 + 状态机）——正确

现在 `openai_client.py` 使用的是 `_strip_think_blocks(buf)` + 累积缓冲区：

```python
# 当前代码（bffcca6 commit 之后）
_think_buf = ""

async for chunk in stream:
    if delta.content:
        _think_buf += delta.content          # 先追加到缓冲区
        visible, _think_buf = _strip_think_blocks(_think_buf)
        if visible:
            yield ApiTextDeltaEvent(text=visible)  # 只 yield 确定可见的部分
```

核心函数 `_strip_think_blocks`：

In [6]:
# 完整复刻 openai_client.py 中的实现
_THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL)

def _strip_think_blocks(buf: str) -> tuple[str, str]:
    """
    Strip complete <think>…</think> blocks from buf and return (visible_text, leftover).

    * Complete <think>…</think> pairs → removed by regex
    * Unclosed <think> at end of buf → held back in leftover
    """
    # 先去掉所有已闭合的 think 块
    cleaned = _THINK_RE.sub('', buf)

    # 如果还有未闭合的 <think>，把它之后的内容全部扣住
    open_idx = cleaned.find('<think>')
    if open_idx != -1:
        return cleaned[:open_idx], cleaned[open_idx:]

    return cleaned, ''


def buffered_strip(chunks: list[str]) -> str:
    """模拟 openai_client.py 的缓冲区方案。"""
    buf = ''
    output_parts = []
    
    for i, chunk in enumerate(chunks):
        buf += chunk
        visible, buf = _strip_think_blocks(buf)
        output_parts.append(visible)
        print(f"  chunk[{i:2d}]: buf after={repr(buf[:40]+'...' if len(buf)>40 else buf)!r:45s}  yield={repr(visible)!r}")
    
    # 流结束后，如果 buf 里还有内容（理论上不应有 <think> 但防御性 flush）
    if buf:
        # 如果流结束了但 <think> 没有闭合（模型截断），丢弃 buf
        print(f"  [flush]  丢弃未闭合的缓冲区: {repr(buf)}")
    
    return ''.join(output_parts)

print("=== 缓冲区方案过程 ===")
buffered_result = buffered_strip(CHUNKS)
print()
print("=== 用户实际看到的内容 ===")
print(buffered_result)
print()
print("包含 <think>？", '<think>' in buffered_result)

=== 缓冲区方案过程 ===
  chunk[ 0]: buf after="'<think>\\n'"                                 yield="''"
  chunk[ 1]: buf after="'<think>\\n用户问的是 Python 中'"                   yield="''"
  chunk[ 2]: buf after="'<think>\\n用户问的是 Python 中的列表推导式。\\n我'"        yield="''"
  chunk[ 3]: buf after="'<think>\\n用户问的是 Python 中的列表推导式。\\n我应该给一个简洁的例...'"  yield="''"
  chunk[ 4]: buf after="''"                                           yield="'\\n列表推导式是'"
  chunk[ 5]: buf after="''"                                           yield="'一种简洁的创'"
  chunk[ 6]: buf after="''"                                           yield="'建列表的'"
  chunk[ 7]: buf after="''"                                           yield="'语法：\\n```python\\nsquares = [x**2 for x in range(10)]\\n```'"

=== 用户实际看到的内容 ===

列表推导式是一种简洁的创建列表的语法：
```python
squares = [x**2 for x in range(10)]
```

包含 <think>？ False


In [14]:
buf = "<think>some reasoning</"
cleaned = _THINK_RE.sub('', buf)
print(cleaned)
# 如果还有未闭合的 <think>，把它之后的内容全部扣住
open_idx = cleaned.find('<think>')
print("open_idx:", open_idx)

<think>some reasoning</
open_idx: 0


In [15]:
buf = "<think>some reasoning</think> xxx"
cleaned = _THINK_RE.sub('', buf)
print(cleaned)
# 如果还有未闭合的 <think>，把它之后的内容全部扣住
open_idx = cleaned.find('<think>')
print("open_idx:", open_idx)

 xxx
open_idx: -1


## 5. 对比总结

In [7]:
EXPECTED = (
    "\n"
    "列表推导式是一种简洁的创建列表的语法：\n"
    "```python\n"
    "squares = [x**2 for x in range(10)]\n"
    "```"
)

print("方案对比")
print("-" * 60)
print(f"期望输出:         {repr(EXPECTED[:50])}...")
print(f"逐chunk正则:      {repr(naive_result[:50])}...")
print(f"缓冲区方案:       {repr(buffered_result[:50])}...")
print()
print(f"逐chunk正则 ✓?   {naive_result.strip() == EXPECTED.strip()}")
print(f"缓冲区方案  ✓?   {buffered_result.strip() == EXPECTED.strip()}")

方案对比
------------------------------------------------------------
期望输出:         '\n列表推导式是一种简洁的创建列表的语法：\n```python\nsquares = [x**2 for'...
逐chunk正则:      '<think>\n用户问的是 Python 中的列表推导式。\n我应该给一个简洁的例子。\n</think'...
缓冲区方案:       '\n列表推导式是一种简洁的创建列表的语法：\n```python\nsquares = [x**2 for'...

逐chunk正则 ✓?   False
缓冲区方案  ✓?   True


## 6. 特殊情况：模型截断（未闭合的 `<think>`）

In [8]:
# 模型在 <think> 中途停止输出（finish_reason=stop / length）
truncated_chunks = [
    "<think>\n",
    "正在思考...",
    # 注意：没有 </think>，流就结束了
]

print("=== 截断场景：<think> 未闭合 ===")
buf = ''
visible_parts = []
for i, chunk in enumerate(truncated_chunks):
    buf += chunk
    visible, buf = _strip_think_blocks(buf)
    visible_parts.append(visible)
    print(f"  chunk[{i}] yield: {repr(visible)!r:20s}  buf: {repr(buf)!r}")

print(f"\n流结束，剩余 buf（丢弃）: {repr(buf)}")
print(f"用户看到的内容: {repr(''.join(visible_parts))!r}")
print("→ 未闭合的 <think> 内容被完整扣住，用户看不到任何推理内容 ✓")

=== 截断场景：<think> 未闭合 ===
  chunk[0] yield: "''"                  buf: "'<think>\\n'"
  chunk[1] yield: "''"                  buf: "'<think>\\n正在思考...'"

流结束，剩余 buf（丢弃）: '<think>\n正在思考...'
用户看到的内容: "''"
→ 未闭合的 <think> 内容被完整扣住，用户看不到任何推理内容 ✓


## 7. 根因总结

| | 逐 chunk 正则替换 | 缓冲区方案（现方案）|
|---|---|---|
| **原理** | 对每个 chunk 独立执行 `re.sub` | 累积所有 chunk，只 yield 已确认可见的前缀 |
| **开标签跨 chunk** | ❌ 两个 chunk 都没有完整配对，全部泄漏 | ✅ 遇到 `<think>` 就停止 yield，等待 `</think>` |
| **闭标签跨 chunk** | ❌ 同上 | ✅ 闭合后 regex 命中，整块被删除 |
| **未闭合（截断）** | ❌ 所有内容都泄漏 | ✅ 流结束时丢弃 buf，用户看不到推理内容 |
| **延迟** | 零延迟 | 微小延迟（持有 `<think>` 到闭合） |

**结论**：流式 `<think>` 过滤必须是**有状态的**（stateful），保持跨 chunk 的上下文。
无状态的逐 chunk 正则替换无法解决标签跨 chunk 边界的问题。

## 8. 真实 LLM 验证：旧方案会直接暴露 `<think>` 内容

下面使用 `~/.openharness/settings.json` 中的 active profile 向真实 LLM 发起流式请求，
分别用**旧方案**（逐 chunk 直接打印）和**新方案**（缓冲区过滤）展示用户实际看到的内容。

> ⚠️ API Key 在运行时从本地配置文件读取，不硬编码在 notebook 中。

In [9]:
import json, re, os
from pathlib import Path
from openai import OpenAI

# ── 读取 ~/.openharness/settings.json 中的 active profile ─────────────────
settings_path = Path.home() / '.openharness' / 'settings.json'
settings = json.loads(settings_path.read_text())

active  = settings.get('active_profile', '')
profile = settings.get('profiles', {}).get(active, {})

BASE_URL = profile.get('base_url') or settings.get('base_url')
MODEL    = profile.get('last_model') or profile.get('default_model') or settings.get('model')
API_KEY  = profile.get('api_key') or settings.get('api_key') or 'placeholder'

print(f'active_profile : {active}')
print(f'base_url       : {BASE_URL}')
print(f'model          : {MODEL}')
print(f'api_key        : {API_KEY[:8]}...{API_KEY[-4:]}')

PROMPT = '用一句话解释什么是递归。'

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# ── 旧方案：逐 chunk 直接打印，不做任何缓冲 ───────────────────────────────
print('\n' + '='*60)
print('【旧方案】逐 chunk 直接输出（用户视角）')
print('='*60)

old_parts = []
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': PROMPT}],
    stream=True,
    max_tokens=4096,   # 足够让 </think> 闭合
)
for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    piece = delta.content or ''
    if piece:
        old_parts.append(piece)
        print(piece, end='', flush=True)

print()
old_full = ''.join(old_parts)
has_think  = '<think>'  in old_full
has_close  = '</think>' in old_full
print(f'\n→ 包含 <think>：{has_think}，包含 </think>：{has_close}，共 {len(old_full)} 字符')
if not has_close:
    print('  ⚠️  </think> 未出现 — 模型在 think 块内被截断，请再增大 max_tokens')


active_profile : coding
base_url       : https://ai-infra.united-imaging.com/v1
model          : MiniMax-M2.5
api_key        : sk-EXaaU...4soM

【旧方案】逐 chunk 直接输出（用户视角）
<think>The user writes in Chinese: "用一句话解释什么是递归。" which translates to "Explain what recursion is in one sentence." The user asks for a single sentence explanation of recursion.

We must respond in Chinese, presumably, as user is Chinese. They want a one-sentence explanation. It can be something like "递归是指在函数的定义中使用函数自身来解决问题的方法，通过将大问题分解为相似的子问题，直到达到基本情况。" That's a single sentence, but might be a bit long but still one sentence. Provide perhaps a concise definition: "递归是指函数直接或间接调用自身，以把问题分解为更小的同类子问题，直到达到基准情形。" That is one sentence. That is okay. However must ensure it's one sentence, no extra period inside the sentence, but can use commas and maybe the Chinese full stop "。" at the end.

We can comply. There's no disallowed content. No policy issues. The user is straightforward, so we answer with one sentence.

We should answe

In [10]:
# ── 新方案：缓冲区过滤 <think>…</think> ────────────────────────────────────
_THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL)

def _strip_think_blocks(buf: str) -> tuple[str, str]:
    cleaned = _THINK_RE.sub('', buf)
    open_idx = cleaned.find('<think>')
    if open_idx != -1:
        return cleaned[:open_idx], cleaned[open_idx:]
    return cleaned, ''

print('\n' + '='*60)
print('【新方案】缓冲区过滤后输出（用户视角）')
print('='*60)

new_parts = []
_buf = ''

stream2 = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': PROMPT}],
    stream=True,
    max_tokens=4096,
)
for chunk in stream2:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    piece = delta.content or ''
    if piece:
        _buf += piece
        visible, _buf = _strip_think_blocks(_buf)
        if visible:
            new_parts.append(visible)
            print(visible, end='', flush=True)

# 流结束：若 buf 非空说明 </think> 从未出现（模型截断），丢弃
if _buf:
    print(f'\n  ⚠️  流结束时缓冲区仍有 {len(_buf)} 字符未闭合，已丢弃')

print()
new_full   = ''.join(new_parts)
has_think2 = '<think>' in new_full
print(f'→ 包含 <think>：{has_think2}，共 {len(new_full)} 字符')

print('\n' + '='*60)
print('对比')
print('='*60)
print(f'旧方案暴露 <think>：{has_think}')
print(f'新方案暴露 <think>：{has_think2}')



【新方案】缓冲区过滤后输出（用户视角）


递归是指在函数或过程内部直接或间接调用自身，通过将原问题分解为相似的子问题，直至子问题足够简单可直接求解的编程技术。
→ 包含 <think>：False，共 60 字符

对比
旧方案暴露 <think>：True
新方案暴露 <think>：False


In [12]:
stream3 = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': PROMPT}],
    stream=False,
    max_tokens=4096,
)

In [13]:
stream3

ChatCompletion(id='c0375749e7c744128037dd2a64c65bf2', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='<think>The user asks: "用一句话解释什么是递归。" (Explain recursion in one sentence). This is a simple request. There\'s no policy violation. Should comply. Provide a concise explanation in Chinese: 递归是指函数直接或间接调用自身来解决问题的过程。 Or "递归是一种在函数定义中使用自身来求解问题的方法，通过将大问题分解为相似的子问题来实现。" But they want one sentence, so a short one: "递归是指一个函数直接或间接调用自身，以把问题分解为更小的相同子问题来求解的方法。" That\'s fine. Provide answer. Should be Chinese.\n\nI\'ll comply.\n</think>\n\n递归是指一个函数直接或间接调用自身，把大问题分解为相同的子问题来求解的过程。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=None), matched_stop=200020)], created=1776665270, model='MiniMax-M2.5', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=143, prompt_tokens=44, total_tokens=187, completion_tokens_details=None, promp